# Basic Decision Tree from Scratch - Classification (Median)

In this notebook, we are going to build a decision tree from scratch using median as a threshold/split point. 

**Please note that in real decision tree we use Gini to compute the threshold for split. In this notebook, we delay that and only use median. The purpose of this notebook is to teach ourselves how to build a tree and make prediction with a tree.**

## Dataset

Dataset: This is a makeup dataset that describe how much study hours a student put in and whether if they will either pass or fail their exam

| Study Hours  | Pass Exam |
| ----- | ----- |
| 1    | No    |
| 2    | No    |
| 3    | No    |
| 4    | No    |
| 5    | Yes   |
| 6    | Yes   |
| 7    | Yes   |
| 8    | Yes   |
| 9    | Yes   |
| 10    | Yes   |
| 11    | Yes   |

In [1]:
import pandas as pd
import json

In [2]:
df = pd.DataFrame({'studied': [1,2,3,4,5,6,7,8,9,10,11,12], 
        'passed': [0,0,0,0,1,1,1,1,1,1,1,1]})

In [3]:
df

,studied,passed
0,1,0
1,2,0
2,3,0
3,4,0
4,5,1
5,6,1
6,7,1
7,8,1
8,9,1
9,10,1


## Tree Building Algorithm (Step by Step)

We are going to create a tree by using median as a cutoff. The steps are as follows:

1. Compute median of feature

2. Split data into:
      - left_node  where values <= median
      - right_node where values > median

3. Condition for stopping split: 
      - If the dataset is empty
      - If there is only one row left in the subset, no more split. 
      - If the node is pure (all the labels in the subset is the same)

4. Continue splitting for non pure node until condition on step 3 satisfied.


### Step 1

In [4]:
median_studied = df['studied'].median()
print(f"Median studied: {median_studied}")

left_subset = df[df['studied'] <= median_studied]
right_subset = df[df['studied'] > median_studied]

print(f"Left subset:\n{left_subset}\n")
print(f"Right subset:\n{right_subset}\n")

Median studied: 6.5
Left subset:
   studied  passed
0        1       0
1        2       0
2        3       0
3        4       0
4        5       1
5        6       1

Right subset:
    studied  passed
6         7       1
7         8       1
8         9       1
9        10       1
10       11       1
11       12       1



- Right node is pure -> no splitting needed.
- Left node is not pure -> need to split further

### Step 2

In [5]:
# right is pure so no splitting needed, but left is not pure so we need to split it further
manual_df = left_subset
median_studied = manual_df['studied'].median()
print(f"Median studied: {median_studied}")

left_subset = manual_df[manual_df['studied'] <= median_studied]
right_subset = manual_df[manual_df['studied'] > median_studied]

print(f"Left subset:\n{left_subset}\n")
print(f"Right subset:\n{right_subset}\n")

Median studied: 3.5
Left subset:
   studied  passed
0        1       0
1        2       0
2        3       0

Right subset:
   studied  passed
3        4       0
4        5       1
5        6       1



- Left node is pure -> no splitting needed.
- Right node is not pure -> need to split further

### Step 3

In [6]:
# left is pure so no splitting needed, but right is not pure so we need to split it further
manual_df = right_subset
median_studied = manual_df['studied'].median()
print(f"Median studied: {median_studied}")

left_subset = manual_df[manual_df['studied'] <= median_studied]
right_subset = manual_df[manual_df['studied'] > median_studied]

print(f"Left subset:\n{left_subset}\n")
print(f"Right subset:\n{right_subset}\n")

Median studied: 5.0
Left subset:
   studied  passed
3        4       0
4        5       1

Right subset:
   studied  passed
5        6       1



- Right node is pure -> no splitting needed.
- Left node is not pure -> need to split further

### Step 4

In [7]:
# right is pure so no splitting needed, but left is not pure so we need to split it further
manual_df = left_subset
median_studied = manual_df['studied'].median()
print(f"Median studied: {median_studied}")

left_subset = manual_df[manual_df['studied'] <= median_studied]
right_subset = manual_df[manual_df['studied'] > median_studied]

print(f"Left subset:\n{left_subset}\n")
print(f"Right subset:\n{right_subset}\n")

Median studied: 4.5
Left subset:
   studied  passed
3        4       0

Right subset:
   studied  passed
4        5       1



Both are single row -> no splitting required. End of tree.

### Create Tree Building Function

We are going to create a tree by using median as a threshold. The steps are as follows:

Build(node_data):

1. Stopping condition:
      - If the dataset is empty:
            - return None
      - If there is only one row left:
            - create leaf and return label
      - If all labels are identical:
            - create leaf and return label

2. Compute median of feature

3. Split rows into:
      - left_node  where values <= median
      - right_node where values > median

4. Create node storing the median threshold

5. Build(left) Recursively

6. Build(right) Recursively

In [8]:
def built_tree(df : pd.DataFrame) -> dict:

    if df.empty:
        return None

    # If there's only one row left, return its label
    if len(df) <= 1:
        #print(f"Single row subset found with label: {int(df['passed'].iloc[0])}")
        return int(df['passed'].iloc[0])
    
    # If all labels are identical, return that label
    # nunique() returns the number of unique values in the 'passed' column
    if df['passed'].nunique() == 1: 
        #print(f"Pure subset found with label: {int(df['passed'].iloc[0])}")
        return int(df['passed'].iloc[0])

    # Calculate the threshold for the 'studied' feature, which is the median value
    median_threshold = df['studied'].median()
    #print(f"Median studied: {median_threshold}")

    # Split the dataset into two subsets based on the threshold
    left_subset = df[df['studied'] <= median_threshold]
    right_subset = df[df['studied'] > median_threshold]

    #print(f"Left subset:\n{left_subset}\n")
    #print(f"Right subset:\n{right_subset}\n")

    # Create a dictionary to represent the current node and store the threshold
    clf = {'median_threshold': float(median_threshold)}

    # Recursively build the left and right subtrees
    clf['left'] = built_tree(left_subset)
    clf['right'] = built_tree(right_subset)

    return clf

### Build Tree 

We build a tree using current data frame.

In [9]:
df

,studied,passed
0,1,0
1,2,0
2,3,0
3,4,0
4,5,1
5,6,1
6,7,1
7,8,1
8,9,1
9,10,1


In [10]:
clf = built_tree(df)

In [11]:
clf


{'median_threshold': 6.5,
 'left': {'median_threshold': 3.5,
  'left': 0,
  'right': {'median_threshold': 5.0,
   'left': {'median_threshold': 4.5, 'left': 0, 'right': 1},
   'right': 1}},
 'right': 1}

In [12]:
# print tree structure in json format
print(json.dumps(clf, indent=8))


{
        "median_threshold": 6.5,
        "left": {
                "median_threshold": 3.5,
                "left": 0,
                "right": {
                        "median_threshold": 5.0,
                        "left": {
                                "median_threshold": 4.5,
                                "left": 0,
                                "right": 1
                        },
                        "right": 1
                }
        },
        "right": 1
}


In [13]:
# print tree structure with guided lines
def print_tree(node, indent="    "):
    if isinstance(node, dict):
        print(f"{indent}Median Threshold: {node['median_threshold']}")
        print(f"{indent}├─ Left Node: <= {node['median_threshold']}")
        print_tree(node['left'], indent + "│   ")
        print(f"{indent}└─ Right Node: > {node['median_threshold']}")
        print_tree(node['right'], indent + "    ")
    else:
        print(f"{indent} ->Label (Pass): {node}")


In [14]:
print_tree(clf)

    Median Threshold: 6.5
    ├─ Left Node: <= 6.5
    │   Median Threshold: 3.5
    │   ├─ Left Node: <= 3.5
    │   │    ->Label (Pass): 0
    │   └─ Right Node: > 3.5
    │       Median Threshold: 5.0
    │       ├─ Left Node: <= 5.0
    │       │   Median Threshold: 4.5
    │       │   ├─ Left Node: <= 4.5
    │       │   │    ->Label (Pass): 0
    │       │   └─ Right Node: > 4.5
    │       │        ->Label (Pass): 1
    │       └─ Right Node: > 5.0
    │            ->Label (Pass): 1
    └─ Right Node: > 6.5
         ->Label (Pass): 1


### Prediction / Inference

To make a prediction we pass the prediction to the tree until we reach a leaf.

In [15]:
# make prediction function
def predict(clf, x):
    node = clf  # start at root of the tree

    # while we are still at a decision node (dictionary)
    while isinstance(node, dict):

        # get the split threshold stored at this node
        threshold = node['median_threshold']

        # go left if x is smaller or equal to threshold
        if x <= threshold:
            print(f"Going left: {x} <= {threshold}")
            node = node['left']
        # otherwise go right
        else:
            print(f"Going right: {x} > {threshold}")
            node = node['right']

    # when we reach a leaf (not a dict), return the prediction
    return node

In [16]:
number_of_hours_studied = 5
prediction = predict(clf, number_of_hours_studied)
print(f"Prediction for student who studied {number_of_hours_studied} hours: {prediction}")

Going left: 5 <= 6.5
Going right: 5 > 3.5
Going left: 5 <= 5.0
Going right: 5 > 4.5
Prediction for student who studied 5 hours: 1


## End